In [1]:
import pandas as pd

In [2]:
data = pd.read_excel('email_classification_dataset.xlsx')

In [3]:
texts = pd.read_csv('email_keywords_extracted.csv')

In [25]:
import ast

texts['lemmatized_tokens'] = texts[
    'lemmatized_tokens'
].apply(
    ast.literal_eval
)

In [26]:
print(
    type(
        texts['lemmatized_tokens'].iloc[0]
    )
)

<class 'list'>


In [27]:
texts['processed_text'] = texts[
    'lemmatized_tokens'
].apply(
    lambda x: " ".join(x)
)

In [28]:
texts[
    [
        'lemmatized_tokens',
        'processed_text'
    ]
].head()

,lemmatized_tokens,processed_text
0,"[charged, service, never, used, please, proces...",charged service never used please process refu...
1,"[account, show, incorrect, subscription, statu...",account show incorrect subscription status sho...
2,"[promised, callback, within, hour, december, o...",promised callback within hour december one cal...
3,"[charged, late, fee, even, though, payment, su...",charged late fee even though payment submitted...
4,"[video, call, platform, drop, exactly, minute,...",video call platform drop exactly minute affect...


In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [30]:
tfidf = TfidfVectorizer(
    max_features=5000
)

In [31]:
X = tfidf.fit_transform(
    texts['processed_text']
)

In [32]:
y = data['priority']

In [33]:
print(y.value_counts())

print()

print(
    y.value_counts(
        normalize=True
    ) * 100
)

priority
High        2175
Medium      1138
Critical     989
Low          798
Name: count, dtype: int64

priority
High        42.647059
Medium      22.313725
Critical    19.392157
Low         15.647059
Name: proportion, dtype: float64


In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [35]:
print(X_train.shape)
print(X_test.shape)

(4080, 579)
(1020, 579)


In [36]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

In [37]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import time

In [38]:
models = {

    "Naive Bayes":
        MultinomialNB(),

    "Logistic Regression":
        LogisticRegression(
            max_iter=1000
        ),

    "Linear SVM":
        LinearSVC(),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=100,
            random_state=42
        )
}

In [39]:
results = []

for name, model in models.items():

    start_time = time.time()

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(
        X_test
    )

    end_time = time.time()

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average='weighted'
    )

    recall = recall_score(
        y_test,
        y_pred,
        average='weighted'
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average='weighted'
    )

    training_time = round(
        end_time - start_time,
        2
    )

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        training_time
    ])

In [40]:
results_df = pd.DataFrame(

    results,

    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "Training Time"
    ]
)

In [41]:
results_df.sort_values(
    by='Accuracy',
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score,Training Time
1,Logistic Regression,0.577451,0.556675,0.577451,0.546360,0.38
2,Linear SVM,0.573529,0.550849,0.573529,0.541404,0.22
3,Random Forest,0.569608,0.548216,0.569608,0.542982,8.42
0,Naive Bayes,0.563725,0.557448,0.563725,0.553981,0.02


In [42]:
model = MultinomialNB()

model.fit(X_train, y_train)

MultinomialNB()

In [43]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

MultinomialNB()

In [44]:
y_pred = model.predict(X_test)

In [45]:
y_pred = nb_model.predict(X_test)

In [46]:
accuracy = nb_model.score(X_test, y_test)

In [47]:
new_emails = [

    "My payment failed and money was deducted",

    "I cannot login to my account",

    "Please refund my order immediately",

    "The application crashes while making payment",

    "Thank you for the excellent support"
]

In [49]:
# Create Cleaning Function
def clean_text(text):

    text = str(text)

    text = text.lower()

    text = re.sub(r'http\S+|www\S+', ' ', text)

    text = re.sub(r'\S+@\S+', ' ', text)

    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    text = text.strip()

    return text

In [57]:
import nltk
import re

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [58]:
processed_emails = []

for email in new_emails:

    cleaned = clean_text(email)

    tokens = word_tokenize(cleaned)

    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]

    lemmas = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    processed_emails.append(
        " ".join(lemmas)
    )

In [59]:
new_vectors = tfidf.transform(
    processed_emails
)

In [60]:
print(new_vectors.shape)

(5, 579)


In [61]:
predictions = nb_model.predict(
    new_vectors
)

In [62]:
for email, prediction in zip(
    new_emails,
    predictions
):

    print("="*80)

    print("Email:")
    print(email)

    print("\nPredicted Category:")
    print(prediction)

    print()

Email:
My payment failed and money was deducted

Predicted Category:
High

Email:
I cannot login to my account

Predicted Category:
High

Email:
Please refund my order immediately

Predicted Category:
High

Email:
The application crashes while making payment

Predicted Category:
High

Email:
Thank you for the excellent support

Predicted Category:
Low



In [63]:
new_vectors = tfidf.transform(processed_emails)

In [65]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(texts['processed_text'])

In [66]:
import joblib

In [67]:
joblib.dump(
    nb_model,
    "naive_bayes_model.pkl"
)

['naive_bayes_model.pkl']

In [68]:
joblib.dump(
    tfidf,
    "tfidf_vectorizer.pkl"
)

['tfidf_vectorizer.pkl']

In [69]:
loaded_model = joblib.load(
    "naive_bayes_model.pkl"
)

In [70]:
loaded_tfidf = joblib.load(
    "tfidf_vectorizer.pkl"
)

In [71]:
sample_email = [
    "My payment failed and money was deducted"
]

In [72]:
sample_vector = loaded_tfidf.transform(
    sample_email
)

In [73]:
prediction = loaded_model.predict(
    sample_vector
)

print(prediction)

['High']


In [74]:
import joblib

loaded_model = joblib.load(
    "naive_bayes_model.pkl"
)

loaded_tfidf = joblib.load(
    "tfidf_vectorizer.pkl"
)

In [75]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import re

In [76]:
stop_words = set(
    stopwords.words('english')
)

lemmatizer = WordNetLemmatizer()

In [77]:
def clean_text(text):

    text = str(text)

    text = text.lower()

    text = re.sub(
        r'http\S+|www\S+',
        ' ',
        text
    )

    text = re.sub(
        r'\S+@\S+',
        ' ',
        text
    )

    text = re.sub(
        r'[^a-zA-Z\s]',
        ' ',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    return text.strip()

In [78]:
def predict_email_category(email):

    cleaned = clean_text(email)

    tokens = word_tokenize(cleaned)

    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]

    lemmas = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    processed_text = " ".join(
        lemmas
    )

    vector = loaded_tfidf.transform(
        [processed_text]
    )

    prediction = loaded_model.predict(
        vector
    )[0]

    return prediction

In [79]:
email = """
My payment failed and money was deducted
"""

result = predict_email_category(
    email
)

print(result)

High


In [80]:
emails = [

    "I cannot login to my account",

    "Please refund my money immediately",

    "The application crashes during payment",

    "Thank you for your excellent service",

    "I have a complaint regarding the support team"
]

In [81]:
for email in emails:

    prediction = predict_email_category(
        email
    )

    print("="*80)

    print("Email:")
    print(email)

    print("\nPrediction:")
    print(prediction)

    print()

Email:
I cannot login to my account

Prediction:
High

Email:
Please refund my money immediately

Prediction:
High

Email:
The application crashes during payment

Prediction:
High

Email:
Thank you for your excellent service

Prediction:
Medium

Email:
I have a complaint regarding the support team

Prediction:
Critical



In [82]:
def predict_with_confidence(email):

    cleaned = clean_text(email)

    tokens = word_tokenize(cleaned)

    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]

    lemmas = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    processed_text = " ".join(
        lemmas
    )

    vector = loaded_tfidf.transform(
        [processed_text]
    )

    prediction = loaded_model.predict(
        vector
    )[0]

    confidence = max(
        loaded_model.predict_proba(
            vector
        )[0]
    )

    return prediction, confidence

In [83]:
category, confidence = predict_with_confidence(
    "My payment failed and money was deducted"
)

print("Category:", category)

print(
    "Confidence:",
    round(confidence*100,2),
    "%"
)

Category: High
Confidence: 71.89 %
